# 第6周 · 作业（预计约5小时）

| 部分 | 内容 | 建议时间 |
| --- | --- | --- |
| A | 第一条正式实验记录：n-gram 受控实验 | 3 小时 |
| B | 生成式 vs 判别式：朴素贝叶斯与逻辑回归的学习曲线（Ng & Jordan 2001） | 1.5 小时 |
| C | 反思 | 30 分钟 |
| D（选做） | digits：PCA 维数对分类准确率的影响（可以用讲义里的 `pca_from_scratch`） | 约 1 小时，不计入 |

**约定**：本周作业不用 AI 编程助手写代码，只让它解释报错信息。

In [ ]:
# 运行这一格来安装课程工具包（每次在 Colab 打开 notebook 后运行一次）
import importlib.util, subprocess, sys
if importlib.util.find_spec("zero2researcher") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ReaganGen/Zero2Researcher"], check=True)
print("课程工具包已就绪")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from zero2researcher import data
from zero2researcher.checks import week06 as ck

## A · 第一条正式实验记录：n-gram 范围

按讲义里的**最小实验单元**标准，做一次受控实验：

- **研究问题**：在 TF-IDF + 逻辑回归基线上，把 n-gram 范围从单词扩大到词对、词三元组，测试准确率会变化吗？
- **自变量**：`ngram_range` ∈ {(1, 1), (1, 2), (1, 3)}；
- **控制变量**：`TfidfVectorizer(min_df=2)`、`LogisticRegression(max_iter=1000)`、测试集固定为课程子集全部 2000 条；
- **"种子"**：每个设置训练 3 次，每次用一个**不同的 3000 条训练子集**：
  `idx = np.random.default_rng(seed).choice(len(train), size=3000, replace=False)`，seed ∈ {0, 1, 2}；
- **报告**：每个设置的测试准确率**均值 ± 标准差**（共 9 次训练），并画一张误差条图。

**关于这里的"种子"**：LBFGS 是确定性的，同一份数据跑几次结果都一样，所以我们用"换一批训练子集"来制造变异。但要清楚它的局限：三个 3000 条的子集都抽自同一个 5000 条的训练集，彼此重叠约六成，并不是独立重复；测试集又固定为同一批 2000 条——单个准确率本身就有约 ±0.8 个百分点的抽样误差（二项分布的标准误 $\sqrt{p(1-p)/n} \approx \sqrt{0.85 \times 0.15 / 2000} \approx 0.008$），而这部分**完全不在你算出的标准差里**。所以这里的标准差会**低估**真实的不确定性：判断"显著"时要格外谨慎，并把这一点写进实验记录的"局限"。（另外，`np.std` 默认 `ddof=0`；只有 3 个数时，样本标准差 `np.std(accs, ddof=1)` 会大约 22%。）

In [ ]:
train = data.load_imdb("train")
test = data.load_imdb("test")

results = {}
# 在这里写你的代码
raise NotImplementedError
for ng, (m, s) in results.items():
    print(f"ngram_range {ng}: {m:.4f} ± {s:.4f}")

In [ ]:
# 在这里写你的代码
raise NotImplementedError


现在把这次实验写成**实验记录**，填满下面的模板（这是你第一条正式记录，之后所有研究作业都用这个格式记录，并按学习指南里的自评表自评）。注意：

- "结果"必须是均值 ± 标准差，不许只报最好的一次；
- 结论写"显著更好 / 显著更差 / 差异小于噪声"三者之一并给出依据——**"差异小于噪声"也是合法发现**。

## 实验记录 #1 · 日期：____

- 研究问题：
- 假设（预期结果，以及什么结果会推翻它）：
- 自变量（这次只改变什么）：
- 控制变量（哪些保持不变；哪些干扰变量单独调优）：
- 基线：
- 数据与划分方式（如何避免泄漏）：
- 随机种子：
- 结果（数值：均值±标准差；图：链接或插图）：
- 解释：
- 局限：
- 下一步：

## B · 生成式 vs 判别式：朴素贝叶斯与逻辑回归的学习曲线

讲义里，在全部 5000 条训练影评上，朴素贝叶斯（0.819）输给了逻辑回归（0.855）。但 Ng & Jordan 的经典论文 [*On Discriminative vs. Generative Classifiers*](https://papers.nips.cc/paper/2001/hash/7b7a53e239400a13bd6be6c91c4f6c4e-Abstract.html)（NIPS 2001）指出，**生成式**模型（朴素贝叶斯：先为每一类建一个"词是怎么出现的"模型）和**判别式**模型（逻辑回归：直接学"给定这些词，是哪一类"）之间有两种截然不同的表现区间：朴素贝叶斯的极限误差更高，但接近它所需的数据少得多——训练数据少时它往往更好，数据多了逻辑回归才反超。在 IMDB 上验证一下：

- **自变量**：训练样本数 n ∈ {50, 100, 200, 500, 1000, 2000, 4000}；
- **种子**：每个 n 抽 10 个不同的训练子集：`idx = np.random.default_rng(seed).choice(len(train), size=n, replace=False)`，seed ∈ {0, 1, …, 9}；
- **特征**：`CountVectorizer(min_df=2)`，**在每个子集上重新 fit**——词表只能来自这 n 条训练影评，就像真实场景里你只有这么多数据；朴素贝叶斯和逻辑回归用同一份特征；
- **模型**：`MultinomialNB()` 与 `LogisticRegression(max_iter=1000)`；
- **测试集**：固定为课程子集的全部 2000 条；
- **结果**：存进字典 `curve_results`，`curve_results[n]` 是形状 (10, 2) 的数组，第 s 行是种子 s 的 (朴素贝叶斯准确率, 逻辑回归准确率)。

一共 140 次训练，在普通笔记本电脑或 Colab 上约半分钟到一分钟。

In [ ]:
sizes = [50, 100, 200, 500, 1000, 2000, 4000]
curve_results = {}
# 在这里写你的代码
raise NotImplementedError
for n in sizes:
    r = curve_results[n]
    print(f"n = {n:4d}   NB {r[:, 0].mean():.3f} ± {r[:, 0].std(ddof=1):.3f}   LR {r[:, 1].mean():.3f} ± {r[:, 1].std(ddof=1):.3f}")
ck.nb_lr_curve(curve_results)

画两张并排的图：左图是两条学习曲线（均值 ± 标准差，x 轴取对数）；右图是**配对差** NB − LR（同一个种子、同一份子集上两个模型的差）的均值 ± 2 倍标准误（标准误 = 配对差的标准差 / √10）。再逐个 n 打印配对差的均值、2 倍标准误和朴素贝叶斯赢的种子数。

和作业 A 一样，这里的标准误会**偏乐观**：n 越大，10 个子集彼此重叠越多（两个子集平均重叠 n/5000，n = 2000 时约四成、n = 4000 时约八成），测试集也是同一批 2000 条，10 个种子并不是独立重复。大 n 时差距远超 2SE，结论不受影响；交叉点附近的判断要更谨慎。

In [ ]:
# 在这里写你的代码
raise NotImplementedError


**问题**（每问两三句）：

1. 对每个 n，结论是"朴素贝叶斯显著更好 / 逻辑回归显著更好 / 差异小于噪声"中的哪一个？依据是配对差的均值是否超出 2 倍标准误，以及朴素贝叶斯赢了几个种子。交叉点大约在哪里？
2. 只用前 3 个种子（`curve_results[n][:3]`）重做 n = 500 的判断，结论是什么？用 10 个种子呢？这说明"至少 3 个种子"这条底线在什么情况下不够用？
3. 你的结果与 Ng & Jordan 的"两种区间"一致吗？用一两句话解释，为什么数据少时朴素贝叶斯占优。
4. （选做思考）如果把词表固定为在全部 5000 条训练影评上 fit 一次，再对每个子集只做 `transform`，你预计小 n 时的结论会怎样？这样做的问题出在哪里？

（在这里回答）

## C · 反思（2–3句话）

"基线阶梯"和"最小实验单元"这两个习惯，哪一个对你将来的实验工作更有冲击？为什么？

（在这里写）

## D（选做，约 1 小时，不计入本周时长）· digits：PCA 维数对分类准确率的影响

把 PCA 当作"信息压缩旋钮"，看保留的维数如何影响下游分类：

- 数据：`load_digits()` 的 64 维原始像素；
- 固定划分：`train_test_split(X, y, random_state=0, stratify=y)`；
- 对每个维数 d ∈ {5, 10, 20, 40}：`Pipeline(PCA(n_components=d, random_state=0), LogisticRegression(max_iter=2000))`，在训练集上 fit、报告测试集准确率；
- 把 4 个准确率画成折线图（x 轴 = 维数），并用 `ck.pca_dims_accuracy` 自查。

想多练一步的话，可以把讲义 S1 的 `pca_from_scratch` 复制过来代替 sklearn 的 PCA：只在训练集上求出 mean 与 components，再把训练集、测试集都投影上去，结果应与 Pipeline 完全相同。

In [ ]:
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline

digits = load_digits()
Xd, yd = digits.data, digits.target
dims = [5, 10, 20, 40]

# 在这里写你的代码
raise NotImplementedError
print(dict(zip(dims, np.round(accs, 4))))
ck.pca_dims_accuracy(accs)

**问题**：对照讲义里的累计解释方差曲线：5 维大约保留了多少方差？这能解释 5 维时准确率的下降吗？从 20 维加到 40 维，收益还剩多少？（三四句话）

（在这里回答）